# PKU-MMD Downloader → Kaggle Dataset

**Mục tiêu:** Tải các folder PKUMMD từ Google Drive về `/kaggle/working/` rồi upload lên Kaggle Dataset theo từng batch (mỗi batch ≤ 15 GB), sau đó xóa file local và lặp lại.

**Vì sao cần batch?** `/kaggle/working/` chỉ có ~19.5 GB. Mỗi batch tạo thành 1 Kaggle Dataset riêng (`batch-001`, `batch-002`...). Khi dùng dữ liệu, mount tất cả batch dataset lại.

**Yêu cầu trước khi chạy:**
- Bật **Internet** trong Kaggle Settings.
- Thêm secret **`RCLONE_TOKEN`** vào Kaggle Secrets (Add-ons → Secrets).

**Thứ tự chạy:** Cell 1 → 2 → 3 → 4 (vòng lặp batch tự động)

In [ ]:
# Cell 1: Cai dat rclone
!curl https://rclone.org/install.sh | bash 2>/dev/null || true
!rclone version | head -1
print("[OK] rclone san sang.")

In [ ]:
# Cell 2: Cau hinh rclone + Kaggle API
import os, json, shutil, subprocess, datetime, re
from urllib.parse import parse_qs, urlparse
from kaggle_secrets import UserSecretsClient

# ── rclone config tu Kaggle Secrets ──
secrets = UserSecretsClient()
RCLONE_TOKEN = secrets.get_secret("RCLONE_TOKEN")

GDRIVE_REMOTE = "gdrive"
rclone_conf_dir = os.path.expanduser("~/.config/rclone")
os.makedirs(rclone_conf_dir, exist_ok=True)
with open(os.path.join(rclone_conf_dir, "rclone.conf"), "w") as f:
    f.write(f"[{GDRIVE_REMOTE}]\ntype = drive\ntoken = {RCLONE_TOKEN}\nscope = drive\n")
print("[OK] rclone config da tao.")

# ── Kiem tra Kaggle API (tu dong co san trong notebook) ──
kaggle_json = os.path.expanduser("~/.kaggle/kaggle.json")
assert os.path.exists(kaggle_json), "Khong tim thay ~/.kaggle/kaggle.json!"
with open(kaggle_json) as f:
    kinfo = json.load(f)
KAGGLE_USERNAME = kinfo["username"]
print(f"[OK] Kaggle API: username = {KAGGLE_USERNAME}")

# ── Utilities ──
def disk_free_gb(path="/kaggle/working"):
    _, _, free = shutil.disk_usage(path)
    return free / 1e9

def dir_size_gb(path):
    total = 0
    for root, _, files in os.walk(path):
        for fname in files:
            try: total += os.path.getsize(os.path.join(root, fname))
            except OSError: pass
    return total / 1e9

def ts():
    return datetime.datetime.now().strftime("%H:%M:%S")

print(f"[OK] Disk hien tai: {disk_free_gb():.1f} GB trong")

In [ ]:
# Cell 3: Cau hinh Google Drive (Dán nguyên Link URL hoac Folder ID)
# =============================================================================

DRIVE_URL = "https://drive.google.com/drive/folders/0B20a4UzO-OyMbUhxV3FzSUlRNXc?resourcekey=0-tlHTUWjVR1nlCkoD70tV4Q"

FOLDER_ID = ""
RESOURCE_KEY = ""

if DRIVE_URL.strip():
    parsed = urlparse(DRIVE_URL.strip())
    match = re.search(r'/folders/([a-zA-Z0-9_-]+)', parsed.path)
    if match:
        FOLDER_ID = match.group(1)
    qs = parse_qs(parsed.query)
    if 'resourcekey' in qs:
        RESOURCE_KEY = qs['resourcekey'][0]

# NẾU BẠN SỬ DỤNG LỐI TẮT (SHORTCUT), hày để trống DRIVE_URL ở trên, và điền tên Lối tắt vào dưới đây:
DRIVE_PATH_FALLBACK = "RGB_VIDEO"
FOLDER_NAME = "RGB_VIDEO"

# ── Cau hinh Kaggle Dataset ──
DATASET_PREFIX       = "pkummd"
DATASET_TITLE_PREFIX = "PKU-MMD"
DATASET_PRIVATE      = True
DATASET_LICENSE      = "other"

# ── Cau hinh batch ──
MAX_BATCH_GB = 15.0

WORK_DIR     = "/kaggle/working/pkummd_batch"
MANIFEST_DIR = "/kaggle/working/manifest"
os.makedirs(WORK_DIR,     exist_ok=True)
os.makedirs(MANIFEST_DIR, exist_ok=True)

MANIFEST_FILE = os.path.join(MANIFEST_DIR, "upload_manifest.json")
if os.path.exists(MANIFEST_FILE):
    with open(MANIFEST_FILE) as f:
        MANIFEST = json.load(f)
    print(f"[OK] Da tai manifest: {MANIFEST_FILE}")
else:
    MANIFEST = {}
    print("[OK] Manifest moi (chua co file nao duoc upload).")

def save_manifest():
    with open(MANIFEST_FILE, "w") as f:
        json.dump(MANIFEST, f, indent=2, ensure_ascii=False)

print(f"\n[Config] Automatic Parsed FOLDER_ID    = '{FOLDER_ID}'")
print(f"[Config] Automatic Parsed RESOURCE_KEY = '{RESOURCE_KEY}'")
print(f"[Config] MAX_BATCH_GB = {MAX_BATCH_GB} GB")
print(f"[Config] Disk trong: {disk_free_gb():.1f} GB")


In [ ]:
# Cell 4: Vong lap chinh - Tai theo batch -> Upload Kaggle Dataset -> Xoa
# ==================================================================================

def get_rclone_cmd_base():
    extra = []
    if FOLDER_ID.strip():
        extra.extend(["--drive-root-folder-id", FOLDER_ID.strip()])
    if RESOURCE_KEY.strip():
        extra.extend(["--drive-resource-key", RESOURCE_KEY.strip()])
    if FOLDER_ID.strip():
        return extra, f"{GDRIVE_REMOTE}:"
    else:
        # fallback to path, e.g. shortcut name
        return extra, f"{GDRIVE_REMOTE}:{DRIVE_PATH_FALLBACK}"

def list_drive_files():
    extra_args, remote_path = get_rclone_cmd_base()
    # Using --files-only to get files and size in bytes
    cmd = ["rclone", "lsf", remote_path, "--format", "ps", "--files-only"] + extra_args
    
    print(f"  Executing: {' '.join(cmd)}")
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
    
    files = []
    for line in r.stdout.strip().splitlines():
        line = line.strip()
        if not line: continue
        parts = line.split(None, 1)
        if len(parts) == 2:
            try: files.append((parts[1].strip(), int(parts[0])))
            except ValueError: pass
            
    if files:
        print(f"  [OK] Tim thay {len(files)} files.")
        return sorted(files, key=lambda x: x[0])
    
    print(f"  [WARN] Khong tim thay file nao hoac co loi!")
    if r.stderr:
        print(f"  [ERROR rclone] {r.stderr[:500]}")
        
    print("  [DEBUG] Thu liet ke root cua Google Drive (de xem ban co shortcut khong):")
    r_debug = subprocess.run(["rclone", "lsd", f"{GDRIVE_REMOTE}:"], capture_output=True, text=True, timeout=30)
    print(r_debug.stdout)
    return []

def make_batches(all_files, done_files_set, max_gb):
    pending = [(f, s) for f, s in all_files if f not in done_files_set]
    batches, cur, cur_gb = [], [], 0.0
    for fname, fsize in pending:
        fgb = fsize / 1e9
        if cur and cur_gb + fgb > max_gb:
            batches.append(cur)
            cur, cur_gb = [], 0.0
        cur.append((fname, fsize))
        cur_gb += fgb
    if cur:
        batches.append(cur)
    return batches, pending

def upload_batch_to_kaggle(local_folder, dataset_slug, title, batch_no):
    meta = {
        "title":   title,
        "id":      f"{KAGGLE_USERNAME}/{dataset_slug}",
        "licenses": [{"name": DATASET_LICENSE}],
        "isPrivate": DATASET_PRIVATE
    }
    meta_path = os.path.join(local_folder, "dataset-metadata.json")
    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=2)

    print(f"    Dang upload len Kaggle Dataset: {KAGGLE_USERNAME}/{dataset_slug}")
    r = subprocess.run(
        ["kaggle", "datasets", "create", "-p", local_folder, "--dir-mode", "tar"],
        capture_output=True, text=True
    )
    os.remove(meta_path)
    if r.returncode == 0:
        print(f"    [OK] Upload thanh cong!")
        return True
    else:
        if "already exists" in r.stderr or "already exists" in r.stdout:
            print(f"    [WARN] Dataset da ton tai, thu tao version moi...")
            meta2 = {"title": title, "id": f"{KAGGLE_USERNAME}/{dataset_slug}",
                     "licenses": [{"name": DATASET_LICENSE}]}
            with open(meta_path, "w") as f:
                json.dump(meta2, f)
            r2 = subprocess.run(
                ["kaggle", "datasets", "version", "-p", local_folder,
                 "-m", f"batch-{batch_no:03d}", "--dir-mode", "tar"],
                capture_output=True, text=True
            )
            os.remove(meta_path)
            if r2.returncode == 0:
                print(f"    [OK] Version moi upload thanh cong!")
                return True
            print(f"    [LOI] kaggle datasets version: {r2.stderr[:300]}")
        else:
            print(f"    [LOI] kaggle datasets create: {r.stderr[:300]}")
        return False


# ══════════════════════════════════════════════════════
# THUC THI
# ══════════════════════════════════════════════════════
print(f"\n{'='*60}")
print(f"[{ts()}] Bat dau xu ly folder: {FOLDER_NAME}")
print(f"{'='*60}")

if FOLDER_NAME not in MANIFEST:
    MANIFEST[FOLDER_NAME] = {"done_files": [], "batch_count": 0}
done_set = set(MANIFEST[FOLDER_NAME]["done_files"])

print(f"  [{ts()}] Dang lay danh sach file tu Drive...")
all_files = list_drive_files()
total_files = len(all_files)
total_gb = sum(s for _, s in all_files) / 1e9
print(f"  Tim thay {total_files} files ({total_gb:.1f} GB tong cong)")
print(f"  Da upload: {len(done_set)}/{total_files} files")

if total_files > 0 and len(done_set) == total_files:
    print(f"  [DONE] Tat ca file da duoc upload! Bo qua folder nay.")
elif total_files > 0:
    batches, pending = make_batches(all_files, done_set, MAX_BATCH_GB)
    print(f"  Con {len(pending)} files chua upload, chia thanh {len(batches)} batch.")
    for i, b in enumerate(batches):
        print(f"    Batch {i+1}: {len(b)} files, {sum(s for _,s in b)/1e9:.1f} GB")

    for batch in batches:
        batch_no = MANIFEST[FOLDER_NAME]["batch_count"] + 1
        batch_gb = sum(s for _, s in batch) / 1e9
        slug = f"{DATASET_PREFIX}-{FOLDER_NAME.lower().replace('_', '-')}-batch-{batch_no:03d}"
        title = f"{DATASET_TITLE_PREFIX} {FOLDER_NAME} Batch {batch_no:03d}"

        print(f"\n  --- Batch {batch_no} | {len(batch)} files | {batch_gb:.1f} GB ---")
        print(f"  Dataset slug: {KAGGLE_USERNAME}/{slug}")

        free = disk_free_gb()
        if free < batch_gb + 1.0:
            print(f"  [WARN] Khong du disk ({free:.1f}GB < {batch_gb+1:.1f}GB). Dang don dep...")
            for f in os.listdir(WORK_DIR):
                fp = os.path.join(WORK_DIR, f)
                if os.path.isfile(fp): os.remove(fp)
            free = disk_free_gb()
            print(f"  Sau don dep: {free:.1f} GB trong")

        print(f"  [{ts()}] Bat dau tai {len(batch)} files tu Drive...")
        filelist_path = "/tmp/rclone_batch.txt"
        with open(filelist_path, "w") as fl:
            for fname, _ in batch:
                fl.write(fname + "\n")

        extra_args, remote_path = get_rclone_cmd_base()
        r = subprocess.run(
            ["rclone", "copy",
             remote_path, WORK_DIR,
             "--files-from", filelist_path,
             "--transfers", "8",
             "--drive-chunk-size", "128M",
             "--progress"] + extra_args,
            capture_output=False, text=True
        )
        if r.returncode != 0:
            print(f"  [LOI] rclone exit={r.returncode}. Bo qua batch nay.")
            continue

        actual_gb = dir_size_gb(WORK_DIR)
        print(f"  [{ts()}] Tai xong! Kich thuoc thuc te: {actual_gb:.1f} GB")

        print(f"  [{ts()}] Bat dau upload len Kaggle...")
        ok = upload_batch_to_kaggle(WORK_DIR, slug, title, batch_no)

        if ok:
            MANIFEST[FOLDER_NAME]["done_files"] += [fname for fname, _ in batch]
            MANIFEST[FOLDER_NAME]["batch_count"] = batch_no
            save_manifest()
            done_set = set(MANIFEST[FOLDER_NAME]["done_files"])
            print(f"  [{ts()}] Manifest da luu ({len(done_set)}/{total_files} files tong).")

            deleted = 0
            for fname, _ in batch:
                fpath = os.path.join(WORK_DIR, fname)
                if os.path.exists(fpath):
                    os.remove(fpath)
                    deleted += 1
            print(f"  [{ts()}] Da xoa {deleted} files. Disk con: {disk_free_gb():.1f} GB")
        else:
            print(f"  [LOI] Upload that bai! Giu lai file, khong xoa. Kiem tra loi roi chay lai.")
            break

print(f"\n{'='*60}")
print(f"[{ts()}] HOAN TAT!")
for folder_name, info in MANIFEST.items():
    print(f"  {folder_name}: {len(info['done_files'])} files | {info['batch_count']} datasets da tao")
print(f"Disk con lai: {disk_free_gb():.1f} GB")


In [ ]:
# Cell 5: Xem ket qua - kiem tra cac dataset da tao tren Kaggle
import subprocess
print("Cac dataset da tao:")
r = subprocess.run(
    ["kaggle", "datasets", "list", "--mine", "--sort-by", "lastUpdated"],
    capture_output=True, text=True
)
for line in r.stdout.splitlines():
    if DATASET_PREFIX.lower() in line.lower() or "ref" in line.lower():
        print(" ", line)

import json, os
MANIFEST_FILE = "/kaggle/working/manifest/upload_manifest.json"
if os.path.exists(MANIFEST_FILE):
    with open(MANIFEST_FILE) as f:
        m = json.load(f)
    print("\nManifest:")
    for folder, info in m.items():
        print(f"  {folder}: {len(info['done_files'])} files, {info['batch_count']} batches")
        for i in range(1, info['batch_count'] + 1):
            slug = f"{DATASET_PREFIX}-{folder.lower().replace('_', '-')}-batch-{i:03d}"
            print(f"    -> kaggle datasets download {KAGGLE_USERNAME}/{slug}")